# Time buckets

### Variables

In [1]:
source = '/tmp/warehouse_oa/*.parquet' # where the query starts
bucket_size_minutes = 1

### Setup

In [ ]:
%%capture
## general setup
%load_ext sql
%sql duckdb://
%config SqlMagic.named_parameters="enabled"
%config SqlMagic.displaylimit = None
import sys
sys.path.append(".")

import warnings
warnings.filterwarnings('ignore')

## sql snippets
## each snippet create a virtual table
# import snippets
from snippets.base_events import get_base_events_snippet
from snippets.bucket_durations import get_bucket_durations_snippet
from snippets.final_durations import get_final_durations_snippet
from snippets.zone_pairs import get_zone_pairs_snippet
from snippets.visit_durations import get_visit_durations_snippet
from snippets.aggregated_data import get_aggregated_data_snippet
from snippets.merged_data import get_merged_data_snippet
from snippets.time_buckets import get_time_buckets_snippet

# generate snippets
base_events_snippet = get_base_events_snippet(source, bucket_size_minutes)
time_buckets_snippet = get_time_buckets_snippet()

bucket_durations_snippet = get_bucket_durations_snippet()
final_durations_snippet = get_final_durations_snippet()
# zone_pairs_snippet = get_zone_pairs_snippet()
# visit_durations_snippet = get_visit_durations_snippet()
aggregated_data_snippet = get_aggregated_data_snippet(bucket_size_minutes)

# save snippets in duckdb
%sql {{base_events_snippet}} --save base_events --no-execute
%sql {{time_buckets_snippet}} --with base_events --save time_buckets --no-execute

%sql {{bucket_durations_snippet}} --with time_buckets  --save bucket_durations --no-execute
%sql {{final_durations_snippet}} --with bucket_durations --save final_durations --no-execute
# %sql {{zone_pairs_snippet}} --with base_events --save zone_pairs --no-execute
# %sql {{visit_durations_snippet}} --with zone_pairs --save visit_durations --no-execute
%sql {{aggregated_data_snippet}} --with final_durations --save aggregated_data --no-execute

_zones = %sql --with visit_durations SELECT DISTINCT zone_name FROM visit_durations
zones = [zone[0] for zone in _zones]

merged_data_snippet = get_merged_data_snippet(zones)
%sql {{merged_data_snippet}} --with aggregated_data --save merged_data --no-execute

### Main

In [3]:
## Get the data - refresh here
data = %sql --with merged_data select * from merged_data LIMIT 100000

data

## plot
from plot import plot2
from bokeh.io import output_notebook

output_notebook()
plot2(data, zones, 'total_visits', bucket_size_minutes)
plot2(data, zones, 'avg_duration_seconds', bucket_size_minutes)
plot2(data, zones, 'total_time', bucket_size_minutes)

# Enable Bokeh outputs in Jupyter Notebook


Running query in 'duckdb://'

Loading BokehJS ...

In [4]:
# %%sql --with merged_data
# select * from merged_data LIMIT 100000